# 02 - Column EDA, cleaning decisions, and CSV export

Notebook này chạy offline từ raw CSV của notebook 01. Thứ tự bắt buộc là: load data gốc -> phân tích từng column/value -> ghi rõ quyết định xử lý -> chạy parser/cleaning rule -> kiểm tra chất lượng -> xuất CSV clean.

Toàn bộ code cleaning nằm trong notebook này; không cần chạy `clean_and_report.py` hay import `utils.clean`.

## Bước 1 - Load raw data

Đây là dữ liệu gốc để phân tích. Raw columns được giữ lại trong output clean để audit về sau.

In [2]:
from pathlib import Path
import json
import re
import unicodedata
from dataclasses import asdict, dataclass, field
from datetime import datetime, timezone
from typing import Any

import pandas as pd

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid')
    PLOTTING_AVAILABLE = True
except ImportError:
    PLOTTING_AVAILABLE = False

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_colwidth', 120)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'notebook':
    NOTEBOOK_DIR = NOTEBOOK_DIR / 'notebook'
DATA_DIR = NOTEBOOK_DIR / 'data'
RAW_PATH = DATA_DIR / 'suumo_parser_records_raw.csv'
LEGACY_RAW_PATH = DATA_DIR / 'suumo_parser_records_eda.csv'
CLEAN_PATH = DATA_DIR / 'suumo_parser_records_clean.csv'
CURRENT_PATH = DATA_DIR / 'suumo_rentals_current.csv'
STATION_PATH = DATA_DIR / 'suumo_station_access.csv'
CURRENT_STATION_PATH = DATA_DIR / 'suumo_station_access_current.csv'
REPORT_PATH = DATA_DIR / 'suumo_clean_report.json'

source_path = RAW_PATH if RAW_PATH.exists() else LEGACY_RAW_PATH
if not source_path.exists():
    raise FileNotFoundError('Missing raw cache. Run 01_pull_and_cache_overview.ipynb first.')

RAW_STRING_COLUMNS = {
    'data_hash': 'string',
    'phone_number': 'string',
    'shop_property_code': 'string',
    'suumo_property_code': 'string',
}
raw = pd.read_csv(source_path, encoding='utf-8-sig', low_memory=False, dtype=RAW_STRING_COLUMNS)
if source_path != RAW_PATH:
    raw.to_csv(RAW_PATH, index=False, encoding='utf-8-sig')
    print(f'Copied legacy raw cache to {RAW_PATH}')
print(f'Raw rows: {len(raw):,}')
print(f'Raw columns: {len(raw.columns):,}')
print(f'Raw path: {RAW_PATH.resolve()}')

Raw rows: 17,927
Raw columns: 57
Raw path: /Users/hoangvinh/Library/CloudStorage/OneDrive-Personal/Workspace/japan_analysis/notebook/data/suumo_parser_records_raw.csv


## Bước 2 - Profile từng column trước khi xử lý

Bảng này cho biết dtype, missing thật, hidden missing sentinel như `-`/`なし`/`不要`, số giá trị khác nhau, và top values an toàn. Các cột nhạy cảm/free-text bị chặn phần sample nhưng vẫn được đếm để hiểu coverage.

In [3]:
NULL_SENTINELS = {'', '-', '―', '－', 'なし', '無し', '未定'}
MONEY_ZERO_SENTINELS = NULL_SENTINELS | {'不要'}
SENSITIVE_COLUMNS = {
    'address', 'address_normalized', 'phone_number', 'phone_number_normalized',
    'remarks', 'guarantee_company', 'other_initial_costs', 'other_monthly_costs',
    'station_access', 'station_access_normalized', 'batch_file_path',
}

def normalize_text_for_profile(value):
    if value is None or pd.isna(value):
        return None
    normalized = unicodedata.normalize('NFKC', str(value)).strip()
    return normalized or None

def safe_top_values(series, top_n=5):
    if series.name in SENSITIVE_COLUMNS:
        return '[blocked from display]'
    counts = series.map(normalize_text_for_profile).value_counts(dropna=False).head(top_n)
    return '; '.join(f'{idx}: {count}' for idx, count in counts.items())

profile_rows = []
for column in raw.columns:
    normalized = raw[column].map(normalize_text_for_profile)
    profile_rows.append({
        'column': column,
        'dtype': str(raw[column].dtype),
        'non_null': int(raw[column].notna().sum()),
        'missing': int(raw[column].isna().sum()),
        'missing_pct': round(raw[column].isna().mean() * 100, 2),
        'sentinel_count': int(normalized.isin(MONEY_ZERO_SENTINELS).sum()),
        'unique_non_null': int(raw[column].nunique(dropna=True)),
        'top_values': safe_top_values(raw[column]),
    })
raw_column_profile = pd.DataFrame(profile_rows).sort_values(['missing_pct', 'sentinel_count'], ascending=[False, False])
raw_column_profile

,column,dtype,non_null,missing,missing_pct,sentinel_count,unique_non_null,top_values
8,batch_started_loading_at,float64,0,17927,100.00,0,0,None: 17927
9,batch_finished_loading_at,float64,0,17927,100.00,0,0,None: 17927
10,batch_loaded_at,float64,0,17927,100.00,0,0,None: 17927
13,error_message,float64,0,17927,100.00,0,0,None: 17927
14,error_type,float64,0,17927,100.00,0,0,None: 17927
15,image_public_url,float64,0,17927,100.00,0,0,None: 17927
16,image_storage_path,float64,0,17927,100.00,0,0,None: 17927
23,brokerage_fee,object,1515,16412,91.55,1296,17,None: 16412; 不要: 1296; 1.1ヶ月: 93; 0.55ヶ月: 85; 1ヶ月: 16
31,contract_period,object,7922,10005,55.81,0,19,None: 10005; 普通借家 2年: 5909; 普通借家 1年: 1662; 定期借家 2年: 242; 定期借家 1年: 30
21,other_monthly_costs,object,9447,8480,47.30,0,2652,[blocked from display]


## Bước 3 - Drill-down từng column khi cần

Đổi `COLUMN_TO_INSPECT` để xem phân phối giá trị của một cột cụ thể trước khi quyết định rule. Hàm này cố tình không cho display các cột nhạy cảm/free-text dài.

In [4]:
def inspect_raw_column(column, top_n=30):
    if column in SENSITIVE_COLUMNS:
        raise ValueError(f'{column} is blocked from notebook display.')
    if column not in raw.columns:
        raise KeyError(f'Unknown column: {column}')
    series = raw[column]
    normalized = series.map(normalize_text_for_profile)
    summary = pd.Series({
        'dtype': str(series.dtype),
        'rows': len(series),
        'non_null': int(series.notna().sum()),
        'missing': int(series.isna().sum()),
        'missing_pct': round(series.isna().mean() * 100, 2),
        'sentinel_count': int(normalized.isin(MONEY_ZERO_SENTINELS).sum()),
        'unique_non_null': int(series.nunique(dropna=True)),
    })
    display(summary.to_frame(column))
    display(normalized.value_counts(dropna=False).head(top_n).to_frame('count'))

COLUMN_TO_INSPECT = 'rent_price_text'
inspect_raw_column(COLUMN_TO_INSPECT)

,rent_price_text
dtype,object
rows,17927
non_null,17927
missing,0
missing_pct,0.0
sentinel_count,0
unique_non_null,409


,count
rent_price_text,
7万円,370
7.1万円,327
7.8万円,305
7.3万円,288
7.5万円,286
8万円,284
8.4万円,280
6万円,269
7.2万円,263


## Bước 4 - Drop cột không dùng phân tích và giải thích cột còn lại

Bước này tạo một dataframe riêng tên `analysis_raw` để phục vụ EDA và thiết kế warehouse. `raw` gốc không bị overwrite vì các bước cleaning phía sau vẫn cần một số cột lineage như `parsed_at`, `batch_created_at`, và `data_hash` để tạo current/history đúng.

Các cột bị drop ở đây thuộc nhóm all-null, operational metadata, đường dẫn file, hoặc dữ liệu nhạy cảm không nên đưa vào mart phân tích.

In [5]:
DROP_COLUMNS_FOR_ANALYSIS = [
    'batch_started_loading_at',
    'batch_finished_loading_at',
    'batch_loaded_at',
    'error_message',
    'error_type',
    'image_public_url',
    'image_storage_path',
    'batch_file_path',
    'batch_status',
    'batch_row_count',
    'batch_inserted_count',
    'batch_failed_count',
    'batch_created_at',
    # 'data_hash',
    'parsed_at',
    'next_update_date_text',
    'phone_number',
]

DROP_COLUMN_REASONS = {
    'batch_started_loading_at': 'Thời điểm loader bắt đầu load batch; snapshot hiện tại 0% non-null, chỉ là metadata vận hành.',
    'batch_finished_loading_at': 'Thời điểm loader kết thúc load batch; snapshot hiện tại 0% non-null, chỉ là metadata vận hành.',
    'batch_loaded_at': 'Thời điểm batch load thành công; snapshot hiện tại 0% non-null, không dùng phân tích listing.',
    'error_message': 'Thông điệp lỗi parser/loader; snapshot hiện tại 0% non-null.',
    'error_type': 'Loại lỗi parser/loader; snapshot hiện tại 0% non-null.',
    'image_public_url': 'URL ảnh public; snapshot hiện tại 0% non-null.',
    'image_storage_path': 'Đường dẫn ảnh trong object storage; snapshot hiện tại 0% non-null.',
    'batch_file_path': 'Đường dẫn file batch trong MinIO; chỉ cần ở raw/staging để audit, không phải thuộc tính listing.',
    'batch_status': 'Trạng thái batch; current snapshot chỉ có một giá trị nên không phân tích được.',
    'batch_row_count': 'Số dòng trong batch; metadata vận hành, không phải thuộc tính căn hộ.',
    'batch_inserted_count': 'Số record batch đã insert; current snapshot chỉ có một giá trị.',
    'batch_failed_count': 'Số record batch lỗi; current snapshot chỉ có một giá trị.',
    'batch_created_at': 'Timestamp batch raw; phần cleaning phía sau sẽ dùng bản typed batch_created_at_utc thay vì cột raw này.',
    # 'data_hash': 'Hash nội dung để trace thay đổi; hữu ích cho staging/history nhưng không cần trong view EDA wide này.',
    'parsed_at': 'Timestamp parser raw; phần cleaning phía sau sẽ dùng parsed_at_utc thay vì cột raw này.',
    'next_update_date_text': 'Ngày cập nhật tiếp theo dạng raw text; phần cleaning dùng next_update_date typed.',
    'phone_number': 'Số điện thoại liên hệ; dữ liệu nhạy cảm/operational, không đưa vào mart phân tích.',
}

missing_drop_columns = [column for column in DROP_COLUMNS_FOR_ANALYSIS if column not in raw.columns]
if missing_drop_columns:
    raise KeyError(f'Các cột muốn drop không tồn tại trong raw: {missing_drop_columns}')

analysis_raw = raw.drop(columns=DROP_COLUMNS_FOR_ANALYSIS).copy()
print(f'Raw columns before drop: {len(raw.columns)}')
print(f'Dropped columns: {len(DROP_COLUMNS_FOR_ANALYSIS)}')
print(f'Columns after drop: {len(analysis_raw.columns)}')

pd.DataFrame({
    'dropped_column': DROP_COLUMNS_FOR_ANALYSIS,
    'reason_vi': [DROP_COLUMN_REASONS[column] for column in DROP_COLUMNS_FOR_ANALYSIS],
})

Raw columns before drop: 57
Dropped columns: 16
Columns after drop: 41


,dropped_column,reason_vi
0,batch_started_loading_at,"Thời điểm loader bắt đầu load batch; snapshot hiện tại 0% non-null, chỉ là metadata vận hành."
1,batch_finished_loading_at,"Thời điểm loader kết thúc load batch; snapshot hiện tại 0% non-null, chỉ là metadata vận hành."
2,batch_loaded_at,"Thời điểm batch load thành công; snapshot hiện tại 0% non-null, không dùng phân tích listing."
3,error_message,Thông điệp lỗi parser/loader; snapshot hiện tại 0% non-null.
4,error_type,Loại lỗi parser/loader; snapshot hiện tại 0% non-null.
5,image_public_url,URL ảnh public; snapshot hiện tại 0% non-null.
6,image_storage_path,Đường dẫn ảnh trong object storage; snapshot hiện tại 0% non-null.
7,batch_file_path,"Đường dẫn file batch trong MinIO; chỉ cần ở raw/staging để audit, không phải thuộc tính listing."
8,batch_status,Trạng thái batch; current snapshot chỉ có một giá trị nên không phân tích được.
9,batch_row_count,"Số dòng trong batch; metadata vận hành, không phải thuộc tính căn hộ."


In [6]:
COLUMN_GROUP_ORDER = {
    'Lineage và định danh': [
        'task_id', 'batch_id', 'source_id', 'suumo_property_code', 'is_valid', 'shop_property_code',
    ],
    'Thời gian, trạng thái giao dịch và vào ở': [
        'information_updated_at_text', 'move_in', 'transaction_type', 'contract_period',
    ],
    'Địa điểm và tiếp cận giao thông': [
        'address', 'station_access',
    ],
    'Giá thuê và chi phí hàng tháng': [
        'rent_price_text', 'management_fee_text', 'other_monthly_costs',
    ],
    'Chi phí ban đầu và bảo lãnh': [
        'deposit_text', 'key_money_text', 'guarantee_deposit_text', 'depreciation_text',
        'brokerage_fee', 'other_initial_costs', 'guarantee_company',
    ],
    'Diện tích, layout, tầng và tuổi nhà': [
        'layout', 'layout_detail', 'exclusive_area_text', 'building_age_text', 'built_at_text',
        'floor_text', 'building_floors', 'total_units',
    ],
    'Loại tòa nhà, kết cấu và hướng': [
        'building_type', 'structure', 'direction',
    ],
    'Điều kiện thuê, bảo hiểm và bãi đỗ xe': [
        'conditions', 'insurance', 'parking',
    ],
    'Năng lượng và tiện ích dự kiến': [
        'energy_efficiency', 'insulation_performance', 'estimated_utility_cost',
    ],
    'Free text cần giữ staging/audit': [
        'remarks',
    ],
}

ordered_analysis_columns = [column for columns in COLUMN_GROUP_ORDER.values() for column in columns]
missing_from_order = sorted(set(analysis_raw.columns) - set(ordered_analysis_columns))
unknown_in_order = sorted(set(ordered_analysis_columns) - set(analysis_raw.columns))
if missing_from_order or unknown_in_order:
    raise ValueError({
        'columns_not_documented': missing_from_order,
        'documented_but_not_in_analysis_raw': unknown_in_order,
    })

analysis_raw = analysis_raw[ordered_analysis_columns]
remaining_columns = pd.DataFrame({
    'order': range(1, len(analysis_raw.columns) + 1),
    'column': analysis_raw.columns,
    'non_null': analysis_raw.notna().sum().values,
    'missing_pct': (analysis_raw.isna().mean() * 100).round(2).values,
    'unique_non_null': [analysis_raw[column].nunique(dropna=True) for column in analysis_raw.columns],
})
remaining_columns

ValueError: {'columns_not_documented': ['data_hash'], 'documented_but_not_in_analysis_raw': []}

In [ ]:
COLUMN_EXPLANATIONS = [
    {'group': 'Lineage và định danh', 'column': 'task_id', 'explanation_vi': 'ID kỹ thuật của task crawl/parser. Dùng để join sang bảng station_access và trace record về task nguồn.', 'warehouse_use': 'Giữ trong mart như technical key.'},
    {'group': 'Lineage và định danh', 'column': 'batch_id', 'explanation_vi': 'ID batch chứa parser record. Hữu ích để trace dữ liệu được sinh từ batch nào, nhưng không phải metric kinh doanh.', 'warehouse_use': 'Giữ staging/mart nếu cần lineage.'},
    {'group': 'Lineage và định danh', 'column': 'source_id', 'explanation_vi': 'ID nguồn dữ liệu trong metadata crawler. Snapshot hiện chỉ có SUUMO nên cột này không phân nhóm được dữ liệu.', 'warehouse_use': 'Có thể bỏ khỏi mart nếu chỉ có một source.'},
    {'group': 'Lineage và định danh', 'column': 'suumo_property_code', 'explanation_vi': 'Mã bất động sản do SUUMO cung cấp. Đây là business key quan trọng để deduplicate và lấy bản ghi current mới nhất.', 'warehouse_use': 'Giữ trong mart.'},
    {'group': 'Lineage và định danh', 'column': 'is_valid', 'explanation_vi': 'Flag parser record hợp lệ. Loader hiện chỉ đưa record hợp lệ vào warehouse nên current snapshot chỉ có true.', 'warehouse_use': 'Bỏ khỏi mart nếu luôn true; giữ staging để audit.'},
    {'group': 'Lineage và định danh', 'column': 'shop_property_code', 'explanation_vi': 'Mã listing theo cửa hàng môi giới. Cardinality cao, có thể thay đổi theo shop và không ổn định bằng suumo_property_code.', 'warehouse_use': 'Giữ staging/audit; không ưu tiên mart phân tích.'},
    {'group': 'Thời gian, trạng thái giao dịch và vào ở', 'column': 'information_updated_at_text', 'explanation_vi': '情報更新日, ngày SUUMO ghi nhận thông tin listing được cập nhật ở dạng text gốc. Cột này giúp đánh giá độ mới của listing trước khi parse sang kiểu date.', 'warehouse_use': 'Giữ staging; mart dùng information_updated_at kiểu date.'},
    {'group': 'Thời gian, trạng thái giao dịch và vào ở', 'column': 'move_in', 'explanation_vi': 'Thông tin khi nào có thể vào ở. Ví dụ 即 là vào ở ngay, 相談 là cần trao đổi, còn dạng 26年9月初旬 là lịch vào ở dự kiến.', 'warehouse_use': 'Giữ staging; mart dùng move_in_status, move_in_month, move_in_period.'},
    {'group': 'Thời gian, trạng thái giao dịch và vào ở', 'column': 'transaction_type', 'explanation_vi': 'Hình thức giao dịch. Ví dụ 仲介 nghĩa là qua môi giới; ảnh hưởng cách hiểu phí môi giới và kênh giao dịch.', 'warehouse_use': 'Giữ staging; mart dùng transaction_type_clean.'},
    {'group': 'Thời gian, trạng thái giao dịch và vào ở', 'column': 'contract_period', 'explanation_vi': 'Thời hạn và loại hợp đồng thuê. Ví dụ 普通借家 là hợp đồng thuê thông thường, 定期借家 là hợp đồng thuê có kỳ hạn cố định.', 'warehouse_use': 'Giữ staging; mart dùng contract_type và contract_years.'},
    {'group': 'Địa điểm và tiếp cận giao thông', 'column': 'address', 'explanation_vi': 'Địa chỉ listing dạng đầy đủ. Có giá trị để parse prefecture/city/ward nhưng không nên hiển thị rộng vì quá chi tiết về vị trí.', 'warehouse_use': 'Giữ staging; mart dùng prefecture/city/ward.'},
    {'group': 'Địa điểm và tiếp cận giao thông', 'column': 'station_access', 'explanation_vi': 'Thông tin tuyến/ga và thời gian đi bộ, bus hoặc xe. Một listing có thể có nhiều dòng ga, nên nên tách thành bảng station_access một-nhiều.', 'warehouse_use': 'Giữ staging; mart dùng primary_station_* và bảng station_access riêng.'},
    {'group': 'Giá thuê và chi phí hàng tháng', 'column': 'rent_price_text', 'explanation_vi': '家賃, tiền thuê nhà hàng tháng ở dạng text gốc như 8.5万円. Đây là giá thuê chính chưa gồm các phí khác.', 'warehouse_use': 'Giữ staging; mart dùng rent_jpy.'},
    {'group': 'Giá thuê và chi phí hàng tháng', 'column': 'management_fee_text', 'explanation_vi': '管理費/共益費, phí quản lý hoặc phí khu vực chung hàng tháng. Ở Nhật khoản này thường trả cùng tiền thuê và cần cộng vào chi phí tháng.', 'warehouse_use': 'Giữ staging; mart dùng management_fee_jpy.'},
    {'group': 'Giá thuê và chi phí hàng tháng', 'column': 'other_monthly_costs', 'explanation_vi': 'ほか諸費用, các chi phí hàng tháng khác ngoài tiền thuê và phí quản lý. Nội dung thường là free text nên chưa parse chắc toàn bộ.', 'warehouse_use': 'Giữ staging/audit; chỉ đưa mart sau khi có rule parse ổn định.'},
    {'group': 'Chi phí ban đầu và bảo lãnh', 'column': 'deposit_text', 'explanation_vi': '敷金, tiền cọc khi thuê nhà. Thường có khả năng hoàn lại một phần/toàn phần tùy hợp đồng, khác với tiền lễ.', 'warehouse_use': 'Giữ staging; mart dùng deposit_jpy.'},
    {'group': 'Chi phí ban đầu và bảo lãnh', 'column': 'key_money_text', 'explanation_vi': '礼金, tiền lễ hay tiền cảm ơn chủ nhà theo tập quán thuê nhà ở Nhật. Khoản này thường không hoàn lại, nên ảnh hưởng mạnh đến chi phí ban đầu.', 'warehouse_use': 'Giữ staging; mart dùng key_money_jpy.'},
    {'group': 'Chi phí ban đầu và bảo lãnh', 'column': 'guarantee_deposit_text', 'explanation_vi': '保証金, tiền bảo đảm. Tùy listing, khoản này có thể gần giống tiền cọc nhưng vẫn nên tách riêng vì ý nghĩa hợp đồng khác nhau.', 'warehouse_use': 'Giữ staging; mart dùng guarantee_deposit_jpy.'},
    {'group': 'Chi phí ban đầu và bảo lãnh', 'column': 'depreciation_text', 'explanation_vi': '敷引/償却, khoản bị trừ hoặc khấu hao từ tiền cọc/tiền bảo đảm. Thường là phần không hoàn lại khi kết thúc hợp đồng.', 'warehouse_use': 'Giữ staging; mart dùng depreciation_jpy.'},
    {'group': 'Chi phí ban đầu và bảo lãnh', 'column': 'brokerage_fee', 'explanation_vi': '仲介手数料, phí môi giới trả cho công ty môi giới. Có thể ghi bằng yen hoặc theo số tháng tiền thuê, ví dụ 1ヶ月.', 'warehouse_use': 'Giữ staging; mart dùng brokerage_fee_estimated_jpy hoặc các cột parsed.'},
    {'group': 'Chi phí ban đầu và bảo lãnh', 'column': 'other_initial_costs', 'explanation_vi': 'ほか初期費用, các chi phí ban đầu khác như thay khóa, vệ sinh, bảo lãnh. Thường là free text có tổng tiền và chi tiết.', 'warehouse_use': 'Giữ staging; mart dùng other_initial_costs_total_jpy nếu parse được.'},
    {'group': 'Chi phí ban đầu và bảo lãnh', 'column': 'guarantee_company', 'explanation_vi': '保証会社, thông tin công ty bảo lãnh thuê nhà và điều kiện phí bảo lãnh. Nội dung dài, high-cardinality, phù hợp audit/NLP hơn là dimension chính.', 'warehouse_use': 'Giữ staging/audit; không ưu tiên mart.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'layout', 'explanation_vi': '間取り, kiểu bố trí phòng như 1R, 1K, 1DK, 1LDK. Đây là dimension quan trọng để so sánh căn hộ cùng phân khúc.', 'warehouse_use': 'Giữ staging; mart dùng layout_normalized và layout_room_count.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'layout_detail', 'explanation_vi': '間取り詳細, chi tiết phòng như kích thước từng phòng. Có thể hữu ích sau này nhưng hiện là text nhiều biến thể.', 'warehouse_use': 'Giữ staging/audit; chưa ưu tiên mart.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'exclusive_area_text', 'explanation_vi': '専有面積, diện tích sử dụng riêng của căn hộ ở dạng text, ví dụ 25.66m2. Đây là diện tích dùng để tính giá thuê trên m2.', 'warehouse_use': 'Giữ staging; mart dùng exclusive_area_m2.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'building_age_text', 'explanation_vi': '築年数, tuổi nhà dạng text. 新築 nghĩa là nhà mới xây; 築99年以上 nghĩa là ít nhất 99 năm chứ không phải giá trị chính xác.', 'warehouse_use': 'Giữ staging; mart dùng building_age_years và building_age_is_lower_bound.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'built_at_text', 'explanation_vi': '築年月, năm/tháng xây dựng. Nếu source chỉ ghi năm thì phải giữ precision để không hiểu nhầm là tháng 1 thật.', 'warehouse_use': 'Giữ staging; mart dùng built_at và built_at_precision.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'floor_text', 'explanation_vi': '階, tầng của căn hộ. Có thể là một tầng cụ thể như 3階 hoặc range như 1-3階 đối với nhà nhiều tầng.', 'warehouse_use': 'Giữ staging; mart dùng floor_min, floor_max, floor_number.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'building_floors', 'explanation_vi': '階建, tổng số tầng của tòa nhà hoặc mô tả kiểu 3階/10階建. Dùng để biết căn hộ nằm trong tòa cao bao nhiêu tầng.', 'warehouse_use': 'Giữ staging; mart dùng building_total_floors.'},
    {'group': 'Diện tích, layout, tầng và tuổi nhà', 'column': 'total_units', 'explanation_vi': '総戸数, tổng số căn/hộ trong tòa nhà nếu source cung cấp. Có thể phản ánh quy mô tòa nhà.', 'warehouse_use': 'Giữ staging; mart dùng total_units_count.'},
    {'group': 'Loại tòa nhà, kết cấu và hướng', 'column': 'building_type', 'explanation_vi': '建物種別, loại bất động sản như mansion/apartment. Ở Nhật マンション thường là chung cư/kết cấu kiên cố hơn apartment nhỏ.', 'warehouse_use': 'Giữ staging; mart dùng building_type_clean.'},
    {'group': 'Loại tòa nhà, kết cấu và hướng', 'column': 'structure', 'explanation_vi': '構造, kết cấu tòa nhà như RC, SRC, thép, gỗ. Có thể ảnh hưởng chất lượng, cách âm, tuổi thọ và giá thuê.', 'warehouse_use': 'Giữ staging; mart dùng structure_clean.'},
    {'group': 'Loại tòa nhà, kết cấu và hướng', 'column': 'direction', 'explanation_vi': '向き, hướng ban công/cửa chính của căn hộ như Nam, Đông, Tây. Có thể ảnh hưởng ánh sáng và giá.', 'warehouse_use': 'Giữ staging; mart dùng direction_clean.'},
    {'group': 'Điều kiện thuê, bảo hiểm và bãi đỗ xe', 'column': 'conditions', 'explanation_vi': '条件, điều kiện thuê như pet, người ở, loại khách thuê, hoặc các điều kiện đặc biệt. Coverage không đầy đủ nên cần dùng thận trọng.', 'warehouse_use': 'Giữ staging; mart dùng conditions_clean nếu cần dimension.'},
    {'group': 'Điều kiện thuê, bảo hiểm và bãi đỗ xe', 'column': 'insurance', 'explanation_vi': '損保, bảo hiểm tài sản/hỏa hoạn khi thuê nhà. Có thể ghi cần bảo hiểm, số tiền, và kỳ hạn như 1.8万円2年.', 'warehouse_use': 'Giữ staging; mart dùng insurance_required, insurance_jpy, insurance_period_years.'},
    {'group': 'Điều kiện thuê, bảo hiểm và bãi đỗ xe', 'column': 'parking', 'explanation_vi': '駐車場, thông tin bãi đỗ xe. Có thể ghi trong khuôn viên, gần đó, khoảng cách và phí theo tháng; dấu - chỉ là source không list parking rõ ràng.', 'warehouse_use': 'Giữ staging; mart dùng parking_listed, parking_type, parking_distance_m, parking_fee_jpy.'},
    {'group': 'Năng lượng và tiện ích dự kiến', 'column': 'energy_efficiency', 'explanation_vi': 'エネルギー消費性能, hiệu suất tiêu thụ năng lượng. Snapshot hiện gần như chỉ có sentinel thiếu dữ liệu nên chưa có giá trị phân tích.', 'warehouse_use': 'Bỏ khỏi mart hiện tại; giữ staging nếu muốn theo dõi khi source có dữ liệu.'},
    {'group': 'Năng lượng và tiện ích dự kiến', 'column': 'insulation_performance', 'explanation_vi': '断熱性能, hiệu suất cách nhiệt. Snapshot hiện gần như không có dữ liệu thực.', 'warehouse_use': 'Bỏ khỏi mart hiện tại; giữ staging nếu muốn theo dõi sau.'},
    {'group': 'Năng lượng và tiện ích dự kiến', 'column': 'estimated_utility_cost', 'explanation_vi': '目安光熱費, chi phí điện/nước/gas tham khảo. Snapshot hiện gần như không có dữ liệu thực.', 'warehouse_use': 'Bỏ khỏi mart hiện tại; giữ staging nếu source bắt đầu cung cấp.'},
    {'group': 'Free text cần giữ staging/audit', 'column': 'remarks', 'explanation_vi': '備考, ghi chú tự do của listing. Có thể chứa nhiều thông tin phụ nhưng noise cao và không nên đưa vào mart wide nếu chưa có NLP/rule parse.', 'warehouse_use': 'Giữ staging/audit; không đưa mart phân tích mặc định.'},
]

column_dictionary = pd.DataFrame(COLUMN_EXPLANATIONS)
if set(column_dictionary['column']) != set(analysis_raw.columns):
    raise ValueError({
        'missing_explanations': sorted(set(analysis_raw.columns) - set(column_dictionary['column'])),
        'extra_explanations': sorted(set(column_dictionary['column']) - set(analysis_raw.columns)),
    })
column_dictionary['column'] = pd.Categorical(
    column_dictionary['column'], categories=ordered_analysis_columns, ordered=True
)
column_dictionary = column_dictionary.sort_values('column').reset_index(drop=True)
column_dictionary.insert(0, 'order', range(1, len(column_dictionary) + 1))

with pd.option_context(
    'display.max_colwidth', None,
    'display.max_columns', None,
    'display.width', None,
):
    display(column_dictionary)

## Bước 5 - Kiểm tra và xử lý từng cột

Phần này đi theo đúng danh sách cột còn lại ở `column_dictionary`. Mỗi cột có 3 block riêng:

1. **Kiểm tra raw**: xem null, sentinel, unique, top values hoặc summary an toàn.
2. **Chạy xử lý**: function xử lý riêng của cột đó nằm ngay trong section của cột, rồi ghi output vào `processed_outputs`.
3. **Kiểm tra sau xử lý**: xem coverage, sample output, và phân phối kết quả đã parse.

Không còn function `process_column()` gom tất cả rule ở một chỗ. Bạn có thể chạy từng cột dần dần, kiểm kết quả, rồi yêu cầu sửa đúng section đang sai.

### Hàm hỗ trợ kiểm tra row và xử lý từng cột

Các hàm dưới đây dùng chung cho toàn bộ section cột. Mỗi hàm có comment/docstring tiếng Việt nói rõ dùng để làm gì. Mặc định preview row sẽ dùng cột an toàn, không in địa chỉ đầy đủ, ghi chú dài, số điện thoại, hoặc free text nhạy cảm.

In [ ]:

# Các cột này không nên hiển thị raw text mặc định trong notebook vì có thể quá chi tiết,
# chứa nội dung tự do, hoặc không cần thiết cho phân tích nhanh.
SENSITIVE_RAW_COLUMNS = {
    # 'address',
    # 'phone_number',
    # 'remarks',
    # 'guarantee_company',
    # 'other_initial_costs',
    # 'other_monthly_costs',
    # 'station_access',
}

# Bộ cột an toàn dùng khi cần xem một vài row khớp điều kiện mà không lộ text nhạy cảm.
SAFE_ROW_PREVIEW_COLUMNS = [
    'task_id',
    'batch_id',
    'suumo_property_code',
    'rent_price_text',
    'management_fee_text',
    'layout',
    'exclusive_area_text',
    'building_age_text',
    'floor_text',
    'building_type',
    'transaction_type',
]

# Dictionary lưu kết quả xử lý từng cột. Mỗi section column sẽ ghi output vào đây.
processed_outputs = {}


def _safe_display_columns(df, display_columns=None, allow_sensitive=False):
    """Dùng để chọn danh sách cột an toàn khi preview row trong notebook."""
    if display_columns is None:
        display_columns = [column for column in SAFE_ROW_PREVIEW_COLUMNS if column in df.columns]
    if allow_sensitive:
        return [column for column in display_columns if column in df.columns]
    return [
        column for column in display_columns
        if column in df.columns and column not in SENSITIVE_RAW_COLUMNS
    ]


def inspect_column_values(df, column, top_n=15):
    """Dùng để kiểm tra một cột trước xử lý: dtype, null, sentinel, unique và top values."""
    if column not in df.columns:
        raise KeyError(f'Không tìm thấy cột: {column}')
    series = df[column]
    normalized = series.map(normalize_text_for_profile)
    summary = pd.Series({
        'dtype': str(series.dtype),
        'rows': len(series),
        'non_null': int(series.notna().sum()),
        'null_count': int(series.isna().sum()),
        'null_pct': round(series.isna().mean() * 100, 2),
        'sentinel_count': int(normalized.isin(MONEY_ZERO_SENTINELS).sum()),
        'unique_non_null': int(series.nunique(dropna=True)),
    })
    display(summary.to_frame(column))

    if column in SENSITIVE_RAW_COLUMNS:
        print(f'Raw top values của {column!r} bị chặn mặc định. Dùng derived output hoặc allow_sensitive=True khi thật sự cần audit.')
        return summary

    display(normalized.value_counts(dropna=False).head(top_n).to_frame('count'))
    return summary


def find_rows_by_value(df, column, value, contains=False, case=False, max_rows=30, display_columns=None, allow_sensitive=False):
    """Dùng để tìm các row có column=value; có thể dùng contains=True để tìm chuỗi chứa value."""
    if column not in df.columns:
        raise KeyError(f'Không tìm thấy cột: {column}')
    series = df[column]
    if contains:
        mask = series.astype('string').str.contains(str(value), case=case, na=False, regex=False)
    else:
        mask = series.map(normalize_text_for_profile) == normalize_text_for_profile(value)

    columns = _safe_display_columns(df, display_columns=display_columns, allow_sensitive=allow_sensitive)
    result = df.loc[mask, columns].head(max_rows).copy()
    print(f'Matched rows: {int(mask.sum()):,}. Showing: {len(result):,}')
    return result


def preview_null_rows(df, column, max_rows=20, display_columns=None, allow_sensitive=False):
    """Dùng để xem sample các row bị null ở một cột, phục vụ kiểm tra missing pattern."""
    if column not in df.columns:
        raise KeyError(f'Không tìm thấy cột: {column}')
    columns = _safe_display_columns(df, display_columns=display_columns, allow_sensitive=allow_sensitive)
    result = df.loc[df[column].isna(), columns].head(max_rows).copy()
    print(f'Null rows: {int(df[column].isna().sum()):,}. Showing: {len(result):,}')
    return result


def _output_summary(output_df):
    """Dùng để tạo bảng coverage nhanh cho dataframe kết quả sau xử lý một cột."""
    return pd.DataFrame({
        'output_column': output_df.columns,
        'non_null': output_df.notna().sum().values,
        'null_pct': (output_df.isna().mean() * 100).round(2).values,
        'unique_non_null': [output_df[column].nunique(dropna=True) for column in output_df.columns],
    })


def check_processed_column(column, top_n=10):
    """Dùng để kiểm tra lại kết quả sau khi đã chạy block xử lý riêng của một cột."""
    if column not in processed_outputs:
        raise KeyError(f'Chưa có output cho {column}. Hãy chạy block xử lý của cột này trước.')
    output_df = processed_outputs[column]
    display(_output_summary(output_df))
    display(output_df.head(10))

    for output_column in output_df.columns:
        series = output_df[output_column]
        if series.nunique(dropna=True) <= 30:
            print(f'\n--- {output_column} top values ---')
            display(series.value_counts(dropna=False).head(top_n).to_frame('count'))
    return output_df

# Ví dụ dùng khi bạn muốn tự drill-down thêm:
# find_rows_by_value(analysis_raw, 'layout', '1K')
# find_rows_by_value(analysis_raw, 'parking', '近隣', contains=True)
# preview_null_rows(analysis_raw, 'contract_period')


### Nhóm: Lineage và định danh

#### `task_id`

ID kỹ thuật của task crawl/parser. Dùng để join sang bảng station_access và trace record về task nguồn.

In [ ]:
# Kiểm tra raw column `task_id` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'task_id')

,task_id
dtype,int64
rows,17927
non_null,17927
null_count,0
null_pct,0.0
sentinel_count,0
unique_non_null,17927


,count
task_id,
21525,1
9695,1
9701,1
9700,1
9699,1
9698,1
9697,1
9696,1
9694,1


dtype              int64
rows               17927
non_null           17927
null_count             0
null_pct             0.0
sentinel_count         0
unique_non_null    17927
dtype: object

In [ ]:
# Function xử lý riêng cho `task_id`: ép ID task về kiểu Int64 để join/trace ổn định.
def process_task_id():
    output_df = pd.DataFrame({'task_id': pd.array(raw['task_id'], dtype='Int64')})
    processed_outputs['task_id'] = output_df
    return output_df.head(10)

process_task_id()

,task_id
0,21525
1,21526
2,21527
3,21528
4,21529
5,21530
6,21531
7,21532
8,21533
9,21534


In [ ]:
# Kiểm tra lại output sau xử lý của column `task_id`.
check_processed_column('task_id')

#### `batch_id`

ID batch chứa parser record. Hữu ích để trace dữ liệu được sinh từ batch nào, nhưng không phải metric kinh doanh.

In [ ]:
# Kiểm tra raw column `batch_id` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'batch_id')

In [ ]:
# Function xử lý riêng cho `batch_id`: ép ID batch về kiểu Int64 để trace batch nguồn.
def process_batch_id():
    output_df = pd.DataFrame({'batch_id': pd.array(raw['batch_id'], dtype='Int64')})
    processed_outputs['batch_id'] = output_df
    return output_df.head(10)

process_batch_id()

,batch_id
0,217
1,217
2,217
3,217
4,217
5,217
6,217
7,217
8,217
9,217


In [ ]:
# Kiểm tra lại output sau xử lý của column `batch_id`.
check_processed_column('batch_id')

#### `source_id`

ID nguồn dữ liệu trong metadata crawler. Snapshot hiện chỉ có SUUMO nên cột này không phân nhóm được dữ liệu.

In [ ]:
# Kiểm tra raw column `source_id` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'source_id')

In [ ]:
# Function xử lý riêng cho `source_id`: ép source id về Int64; hiện chỉ có một source SUUMO.
def process_source_id():
    output_df = pd.DataFrame({'source_id': pd.array(raw['source_id'], dtype='Int64')})
    processed_outputs['source_id'] = output_df
    return output_df.head(10)

process_source_id()

,source_id
0,1
1,1
2,1
3,1
4,1
5,1
6,1
7,1
8,1
9,1


In [ ]:
# Kiểm tra lại output sau xử lý của column `source_id`.
check_processed_column('source_id')

#### `suumo_property_code`

Mã bất động sản do SUUMO cung cấp. Đây là business key quan trọng để deduplicate và lấy bản ghi current mới nhất.

In [ ]:
# Kiểm tra raw column `suumo_property_code` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'suumo_property_code')

,suumo_property_code
dtype,string
rows,17927
non_null,17927
null_count,0
null_pct,0.0
sentinel_count,0
unique_non_null,17380


,count
suumo_property_code,
100513148551,3
100513148570,3
100513148629,3
100513148636,3
100518256547,3
100513334320,2
100513334567,2
100513334430,2
100513334296,2


dtype              string
rows                17927
non_null            17927
null_count              0
null_pct              0.0
sentinel_count          0
unique_non_null     17380
dtype: object

In [29]:
find_rows_by_value(analysis_raw, 'suumo_property_code', '100513148551', display_columns=['task_id', 'batch_id', 'suumo_property_code', 'data_hash'])

Matched rows: 3. Showing: 3


,task_id,batch_id,suumo_property_code,data_hash
457,21189,213,100513148551,d53fac2d33b38924516a211b7e33a3464d3fa6e83171bea5cc962543805f45fe
10207,11354,115,100513148551,c01244502e2736f131f2619f4968b2a2fbc93f7be7485e007e69e0e6a0acfb85
14404,7230,73,100513148551,265c836f5a77a9b5c7c02cb01f320d4771d8574c904adeab44a71b69c9f7ed21


In [ ]:
# Function xử lý riêng cho `suumo_property_code`: normalize text để giữ business key ổn định.
def process_suumo_property_code():
    output_df = pd.DataFrame({'suumo_property_code': normalize_text_series(raw['suumo_property_code'])})
    processed_outputs['suumo_property_code'] = output_df
    return output_df.head(10)

process_suumo_property_code()

In [ ]:
# Kiểm tra lại output sau xử lý của column `suumo_property_code`.
check_processed_column('suumo_property_code')

#### `is_valid`

Flag parser record hợp lệ. Loader hiện chỉ đưa record hợp lệ vào warehouse nên current snapshot chỉ có true.

In [1]:
# Kiểm tra raw column `is_valid` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'is_valid')

NameError: name 'inspect_column_values' is not defined

In [ ]:
# Function xử lý riêng cho `is_valid`: ép về boolean để kiểm tra loader có lọc invalid đúng không.
def process_is_valid():
    output_df = pd.DataFrame({'is_valid': raw['is_valid'].astype('boolean')})
    processed_outputs['is_valid'] = output_df
    return output_df.head(10)

process_is_valid()

In [ ]:
# Kiểm tra lại output sau xử lý của column `is_valid`.
check_processed_column('is_valid')

#### `shop_property_code`

Mã listing theo cửa hàng môi giới. Cardinality cao, có thể thay đổi theo shop và không ổn định bằng suumo_property_code.

In [ ]:
# Kiểm tra raw column `shop_property_code` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'shop_property_code')

In [ ]:
# Function xử lý riêng cho `shop_property_code`: normalize mã listing của shop và đổi sentinel thành null.
def process_shop_property_code():
    output_df = pd.DataFrame({'shop_property_code_clean': clean_categorical(raw['shop_property_code'])})
    processed_outputs['shop_property_code'] = output_df
    return output_df.head(10)

process_shop_property_code()

In [ ]:
# Kiểm tra lại output sau xử lý của column `shop_property_code`.
check_processed_column('shop_property_code')

### Nhóm: Thời gian, trạng thái giao dịch và vào ở

#### `information_updated_at_text`

Ngày SUUMO ghi nhận thông tin listing được cập nhật, ở dạng text gốc. Sau cleaning nên dùng information_updated_at kiểu date.

In [ ]:
# Kiểm tra raw column `information_updated_at_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'information_updated_at_text')

In [ ]:
# Function xử lý riêng cho `information_updated_at_text`: parse ngày cập nhật thông tin từ text sang date.
def process_information_updated_at_text():
    output_df = pd.DataFrame({
        'information_updated_at': pd.to_datetime(raw['information_updated_at_text'], format='%Y/%m/%d', errors='coerce')
    })
    processed_outputs['information_updated_at_text'] = output_df
    return output_df.head(10)

process_information_updated_at_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `information_updated_at_text`.
check_processed_column('information_updated_at_text')

#### `move_in`

Thông tin khi nào có thể vào ở. Ví dụ 即 là vào ở ngay, 相談 là cần trao đổi, còn dạng 26年9月初旬 là lịch vào ở dự kiến.

In [ ]:
# Kiểm tra raw column `move_in` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'move_in')

In [ ]:
# Function xử lý riêng cho `move_in`: tách trạng thái vào ở, tháng dự kiến, và giai đoạn đầu/giữa/cuối tháng.
def process_move_in():
    parsed_at_year = pd.to_datetime(raw['parsed_at'], format='mixed', errors='coerce', utc=True).dt.year
    parsed = pd.Series([
        parse_move_in(value, int(reference_year) if pd.notna(reference_year) else None)
        for value, reference_year in zip(raw['move_in'], parsed_at_year)
    ], index=raw.index)
    output_df = pd.DataFrame({
        'move_in_status': parsed.map(lambda value: value[0]).astype('string'),
        'move_in_month': parsed.map(lambda value: value[1]),
        'move_in_period': parsed.map(lambda value: value[2]).astype('string'),
    })
    processed_outputs['move_in'] = output_df
    return output_df.head(10)

process_move_in()

In [ ]:
# Kiểm tra lại output sau xử lý của column `move_in`.
check_processed_column('move_in')

#### `transaction_type`

Hình thức giao dịch. Ví dụ 仲介 nghĩa là qua môi giới; ảnh hưởng cách hiểu phí môi giới và kênh giao dịch.

In [ ]:
# Kiểm tra raw column `transaction_type` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'transaction_type')

In [ ]:
# Function xử lý riêng cho `transaction_type`: normalize hình thức giao dịch như 仲介 và đổi sentinel thành null.
def process_transaction_type():
    output_df = pd.DataFrame({'transaction_type_clean': clean_categorical(raw['transaction_type'])})
    processed_outputs['transaction_type'] = output_df
    return output_df.head(10)

process_transaction_type()

In [ ]:
# Kiểm tra lại output sau xử lý của column `transaction_type`.
check_processed_column('transaction_type')

#### `contract_period`

Thời hạn và loại hợp đồng thuê. Ví dụ 普通借家 là hợp đồng thuê thông thường, 定期借家 là hợp đồng thuê có kỳ hạn cố định.

In [ ]:
# Kiểm tra raw column `contract_period` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'contract_period')

In [ ]:
# Function xử lý riêng cho `contract_period`: parse loại hợp đồng thuê và số năm hợp đồng.
def process_contract_period():
    parsed = raw['contract_period'].map(parse_contract)
    output_df = pd.DataFrame({
        'contract_type': parsed.map(lambda value: value[0]).astype('string'),
        'contract_years': pd.array(parsed.map(lambda value: value[1]), dtype='Int64'),
    })
    processed_outputs['contract_period'] = output_df
    return output_df.head(10)

process_contract_period()

In [ ]:
# Kiểm tra lại output sau xử lý của column `contract_period`.
check_processed_column('contract_period')

### Nhóm: Địa điểm và tiếp cận giao thông

#### `address`

Địa chỉ listing dạng đầy đủ. Có giá trị để parse prefecture/city/ward nhưng không nên hiển thị rộng vì quá chi tiết về vị trí.

In [ ]:
# Kiểm tra raw column `address` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'address')

In [ ]:
# Function xử lý riêng cho `address`: chỉ parse prefecture/city/ward, không hiển thị địa chỉ đầy đủ.
def process_address():
    parsed = raw['address'].map(parse_address_parts)
    output_df = pd.DataFrame({
        'prefecture': parsed.map(lambda value: value[0]).astype('string'),
        'city': parsed.map(lambda value: value[1]).astype('string'),
        'ward': parsed.map(lambda value: value[2]).astype('string'),
    })
    processed_outputs['address'] = output_df
    return output_df.head(10)

process_address()

In [ ]:
# Kiểm tra lại output sau xử lý của column `address`.
check_processed_column('address')

#### `station_access`

Thông tin tuyến/ga và thời gian đi bộ, bus hoặc xe. Một listing có thể có nhiều dòng ga, nên nên tách thành bảng station_access một-nhiều.

In [ ]:
# Kiểm tra raw column `station_access` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'station_access')

In [ ]:
# Function xử lý riêng cho `station_access`: split route ga/tuyến, lấy primary route và đếm số route.
def process_station_access():
    temp = raw[['task_id', 'batch_id', 'suumo_property_code', 'station_access']].copy()
    rows = []
    failures_by_task = {}
    for _, record in temp.iterrows():
        record_rows, failures = _station_rows_for_record(record)
        rows.extend(record_rows)
        failures_by_task[record['task_id']] = failures
    station_df = pd.DataFrame(rows)
    output_df = pd.DataFrame(index=raw.index)
    output_df['station_parse_failure_count'] = raw['task_id'].map(failures_by_task).fillna(0).astype('Int64')
    if not station_df.empty:
        primary = station_df.sort_values(['task_id', 'access_order']).drop_duplicates('task_id').set_index('task_id')
        counts = station_df.groupby('task_id').size()
        output_df['primary_station_line'] = raw['task_id'].map(primary['station_line'])
        output_df['primary_station_name'] = raw['task_id'].map(primary['station_name'])
        output_df['primary_access_mode'] = raw['task_id'].map(primary['access_mode'])
        output_df['primary_total_travel_minutes'] = pd.array(raw['task_id'].map(primary['total_travel_minutes']), dtype='Int64')
        output_df['station_count'] = pd.array(raw['task_id'].map(counts), dtype='Int64')
    processed_outputs['station_access'] = output_df
    return output_df.head(10)

process_station_access()

In [ ]:
# Kiểm tra lại output sau xử lý của column `station_access`.
check_processed_column('station_access')

### Nhóm: Giá thuê và chi phí hàng tháng

#### `rent_price_text`

家賃, tiền thuê nhà hàng tháng ở dạng text gốc như 8.5万円. Đây là giá thuê chính chưa gồm các phí khác.

In [ ]:
# Kiểm tra raw column `rent_price_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'rent_price_text')

In [ ]:
# Function xử lý riêng cho `rent_price_text`: parse 家賃 từ 万円/円 sang yen Nhật.
def process_rent_price_text():
    output_df = pd.DataFrame({'rent_jpy': pd.array(raw['rent_price_text'].map(parse_jpy_value), dtype='Int64')})
    processed_outputs['rent_price_text'] = output_df
    return output_df.head(10)

process_rent_price_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `rent_price_text`.
check_processed_column('rent_price_text')

#### `management_fee_text`

管理費/共益費, phí quản lý hoặc phí khu vực chung hàng tháng. Ở Nhật khoản này thường trả cùng tiền thuê và cần cộng vào chi phí tháng.

In [ ]:
# Kiểm tra raw column `management_fee_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'management_fee_text')

In [ ]:
# Function xử lý riêng cho `management_fee_text`: parse 管理費/共益費; sentinel như '-' được hiểu là 0.
def process_management_fee_text():
    output_df = pd.DataFrame({
        'management_fee_jpy': pd.array(raw['management_fee_text'].map(lambda value: parse_jpy_value(value, sentinel_as_zero=True)), dtype='Int64')
    })
    processed_outputs['management_fee_text'] = output_df
    return output_df.head(10)

process_management_fee_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `management_fee_text`.
check_processed_column('management_fee_text')

#### `other_monthly_costs`

ほか諸費用, các chi phí hàng tháng khác ngoài tiền thuê và phí quản lý. Nội dung thường là free text nên chưa parse chắc toàn bộ.

In [ ]:
# Kiểm tra raw column `other_monthly_costs` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'other_monthly_costs')

In [ ]:
# Function xử lý riêng cho `other_monthly_costs`: không parse tiền vội; tạo flag có text và độ dài text để audit.
def process_other_monthly_costs():
    normalized = normalize_text_series(raw['other_monthly_costs'])
    output_df = pd.DataFrame({
        'other_monthly_costs_present': normalized.notna() & ~normalized.isin(NULL_SENTINELS),
        'other_monthly_costs_text_length': normalized.str.len().astype('Int64'),
    })
    processed_outputs['other_monthly_costs'] = output_df
    return output_df.head(10)

process_other_monthly_costs()

In [ ]:
# Kiểm tra lại output sau xử lý của column `other_monthly_costs`.
check_processed_column('other_monthly_costs')

### Nhóm: Chi phí ban đầu và bảo lãnh

#### `deposit_text`

敷金, tiền cọc khi thuê nhà. Thường có khả năng hoàn lại một phần/toàn phần tùy hợp đồng, khác với tiền lễ.

In [ ]:
# Kiểm tra raw column `deposit_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'deposit_text')

In [ ]:
# Function xử lý riêng cho `deposit_text`: parse 敷金 tiền cọc; sentinel như '-' được hiểu là 0.
def process_deposit_text():
    output_df = pd.DataFrame({'deposit_jpy': pd.array(raw['deposit_text'].map(lambda value: parse_jpy_value(value, sentinel_as_zero=True)), dtype='Int64')})
    processed_outputs['deposit_text'] = output_df
    return output_df.head(10)

process_deposit_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `deposit_text`.
check_processed_column('deposit_text')

#### `key_money_text`

礼金, tiền lễ hay tiền cảm ơn chủ nhà theo tập quán thuê nhà ở Nhật. Khoản này thường không hoàn lại, nên ảnh hưởng mạnh đến chi phí ban đầu.

In [ ]:
# Kiểm tra raw column `key_money_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'key_money_text')

In [ ]:
# Function xử lý riêng cho `key_money_text`: parse 礼金 tiền lễ/tền cảm ơn; sentinel như '-' được hiểu là 0.
def process_key_money_text():
    output_df = pd.DataFrame({'key_money_jpy': pd.array(raw['key_money_text'].map(lambda value: parse_jpy_value(value, sentinel_as_zero=True)), dtype='Int64')})
    processed_outputs['key_money_text'] = output_df
    return output_df.head(10)

process_key_money_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `key_money_text`.
check_processed_column('key_money_text')

#### `guarantee_deposit_text`

保証金, tiền bảo đảm. Tùy listing, khoản này có thể gần giống tiền cọc nhưng vẫn nên tách riêng vì ý nghĩa hợp đồng khác nhau.

In [ ]:
# Kiểm tra raw column `guarantee_deposit_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'guarantee_deposit_text')

In [ ]:
# Function xử lý riêng cho `guarantee_deposit_text`: parse 保証金 tiền bảo đảm; sentinel như '-' được hiểu là 0.
def process_guarantee_deposit_text():
    output_df = pd.DataFrame({'guarantee_deposit_jpy': pd.array(raw['guarantee_deposit_text'].map(lambda value: parse_jpy_value(value, sentinel_as_zero=True)), dtype='Int64')})
    processed_outputs['guarantee_deposit_text'] = output_df
    return output_df.head(10)

process_guarantee_deposit_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `guarantee_deposit_text`.
check_processed_column('guarantee_deposit_text')

#### `depreciation_text`

敷引/償却, khoản bị trừ hoặc khấu hao từ tiền cọc/tiền bảo đảm. Thường là phần không hoàn lại khi kết thúc hợp đồng.

In [ ]:
# Kiểm tra raw column `depreciation_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'depreciation_text')

In [ ]:
# Function xử lý riêng cho `depreciation_text`: parse 敷引/償却 khoản không hoàn lại/khấu hao; sentinel như '-' được hiểu là 0.
def process_depreciation_text():
    output_df = pd.DataFrame({'depreciation_jpy': pd.array(raw['depreciation_text'].map(lambda value: parse_jpy_value(value, sentinel_as_zero=True)), dtype='Int64')})
    processed_outputs['depreciation_text'] = output_df
    return output_df.head(10)

process_depreciation_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `depreciation_text`.
check_processed_column('depreciation_text')

#### `brokerage_fee`

仲介手数料, phí môi giới trả cho công ty môi giới. Có thể ghi bằng yen hoặc theo số tháng tiền thuê, ví dụ 1ヶ月.

In [ ]:
# Kiểm tra raw column `brokerage_fee` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'brokerage_fee')

In [ ]:
# Function xử lý riêng cho `brokerage_fee`: parse 仲介手数料 theo yen hoặc theo số tháng tiền thuê, rồi ước tính yen.
def process_brokerage_fee():
    brokerage_jpy = raw['brokerage_fee'].map(lambda value: parse_jpy_value(value, sentinel_as_zero=True))
    brokerage_months = raw['brokerage_fee'].map(parse_month_value)
    rent_jpy = raw['rent_price_text'].map(parse_jpy_value)
    estimated = pd.Series(brokerage_jpy).where(pd.Series(brokerage_jpy).notna(), pd.Series(brokerage_months) * pd.Series(rent_jpy))
    output_df = pd.DataFrame({
        'brokerage_fee_jpy': pd.array(brokerage_jpy, dtype='Int64'),
        'brokerage_fee_months': pd.array(brokerage_months, dtype='Float64'),
        'brokerage_fee_estimated_jpy': pd.array(estimated.round(), dtype='Int64'),
    })
    processed_outputs['brokerage_fee'] = output_df
    return output_df.head(10)

process_brokerage_fee()

In [ ]:
# Kiểm tra lại output sau xử lý của column `brokerage_fee`.
check_processed_column('brokerage_fee')

#### `other_initial_costs`

ほか初期費用, các chi phí ban đầu khác như thay khóa, vệ sinh, bảo lãnh. Thường là free text có tổng tiền và chi tiết.

In [ ]:
# Kiểm tra raw column `other_initial_costs` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'other_initial_costs')

In [ ]:
# Function xử lý riêng cho `other_initial_costs`: lấy flag/text length và parse tổng chi phí ban đầu nếu có dạng 合計...円/万円.
def process_other_initial_costs():
    normalized = normalize_text_series(raw['other_initial_costs'])
    output_df = pd.DataFrame({
        'other_initial_costs_present': normalized.notna() & ~normalized.isin(NULL_SENTINELS),
        'other_initial_costs_text_length': normalized.str.len().astype('Int64'),
        'other_initial_costs_total_jpy': pd.array(raw['other_initial_costs'].map(parse_initial_cost_total), dtype='Int64'),
    })
    processed_outputs['other_initial_costs'] = output_df
    return output_df.head(10)

process_other_initial_costs()

In [ ]:
# Kiểm tra lại output sau xử lý của column `other_initial_costs`.
check_processed_column('other_initial_costs')

#### `guarantee_company`

保証会社, thông tin công ty bảo lãnh thuê nhà và điều kiện phí bảo lãnh. Nội dung dài, high-cardinality, phù hợp audit/NLP hơn là dimension chính.

In [ ]:
# Kiểm tra raw column `guarantee_company` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'guarantee_company')

In [ ]:
# Function xử lý riêng cho `guarantee_company`: không parse dimension vội; tạo flag có text và độ dài text để audit.
def process_guarantee_company():
    normalized = normalize_text_series(raw['guarantee_company'])
    output_df = pd.DataFrame({
        'guarantee_company_present': normalized.notna() & ~normalized.isin(NULL_SENTINELS),
        'guarantee_company_text_length': normalized.str.len().astype('Int64'),
    })
    processed_outputs['guarantee_company'] = output_df
    return output_df.head(10)

process_guarantee_company()

In [ ]:
# Kiểm tra lại output sau xử lý của column `guarantee_company`.
check_processed_column('guarantee_company')

### Nhóm: Diện tích, layout, tầng và tuổi nhà

#### `layout`

間取り, kiểu bố trí phòng như 1R, 1K, 1DK, 1LDK. Đây là dimension quan trọng để so sánh căn hộ cùng phân khúc.

In [ ]:
# Kiểm tra raw column `layout` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'layout')

In [ ]:
# Function xử lý riêng cho `layout`: normalize 間取り, tách số phòng và flag service room.
def process_layout():
    parsed = raw['layout'].map(parse_layout)
    output_df = pd.DataFrame({
        'layout_normalized': parsed.map(lambda value: value[0]).astype('string'),
        'layout_room_count': pd.array(parsed.map(lambda value: value[1]), dtype='Int64'),
        'layout_has_service_room': pd.array(parsed.map(lambda value: value[2]), dtype='boolean'),
    })
    processed_outputs['layout'] = output_df
    return output_df.head(10)

process_layout()

In [ ]:
# Kiểm tra lại output sau xử lý của column `layout`.
check_processed_column('layout')

#### `layout_detail`

間取り詳細, chi tiết phòng như kích thước từng phòng. Có thể hữu ích sau này nhưng hiện là text nhiều biến thể.

In [ ]:
# Kiểm tra raw column `layout_detail` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'layout_detail')

In [ ]:
# Function xử lý riêng cho `layout_detail`: normalize chi tiết layout và đổi sentinel thành null.
def process_layout_detail():
    output_df = pd.DataFrame({'layout_detail_clean': clean_categorical(raw['layout_detail'])})
    processed_outputs['layout_detail'] = output_df
    return output_df.head(10)

process_layout_detail()

In [ ]:
# Kiểm tra lại output sau xử lý của column `layout_detail`.
check_processed_column('layout_detail')

#### `exclusive_area_text`

専有面積, diện tích sử dụng riêng của căn hộ ở dạng text, ví dụ 25.66m2. Đây là diện tích dùng để tính giá thuê trên m2.

In [ ]:
# Kiểm tra raw column `exclusive_area_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'exclusive_area_text')

In [ ]:
# Function xử lý riêng cho `exclusive_area_text`: parse 専有面積 từ m2/m² sang số thực.
def process_exclusive_area_text():
    output_df = pd.DataFrame({'exclusive_area_m2': pd.array(raw['exclusive_area_text'].map(parse_area_m2), dtype='Float64')})
    processed_outputs['exclusive_area_text'] = output_df
    return output_df.head(10)

process_exclusive_area_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `exclusive_area_text`.
check_processed_column('exclusive_area_text')

#### `building_age_text`

築年数, tuổi nhà dạng text. 新築 nghĩa là nhà mới xây; 築99年以上 nghĩa là ít nhất 99 năm chứ không phải giá trị chính xác.

In [ ]:
# Kiểm tra raw column `building_age_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'building_age_text')

In [ ]:
# Function xử lý riêng cho `building_age_text`: parse 築年数; 新築 = 0, 築99年以上 là lower bound.
def process_building_age_text():
    normalized = normalize_text_series(raw['building_age_text'])
    output_df = pd.DataFrame({
        'building_age_years': pd.array(raw['building_age_text'].map(parse_building_age), dtype='Int64'),
        'building_age_is_lower_bound': normalized.str.endswith('以上', na=False).astype('boolean'),
    })
    processed_outputs['building_age_text'] = output_df
    return output_df.head(10)

process_building_age_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `building_age_text`.
check_processed_column('building_age_text')

#### `built_at_text`

築年月, năm/tháng xây dựng. Nếu source chỉ ghi năm thì phải giữ precision để không hiểu nhầm là tháng 1 thật.

In [ ]:
# Kiểm tra raw column `built_at_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'built_at_text')

In [ ]:
# Function xử lý riêng cho `built_at_text`: parse 築年月 và ghi precision year/month.
def process_built_at_text():
    normalized = normalize_text_series(raw['built_at_text'])
    precision = pd.Series(pd.NA, index=raw.index, dtype='string')
    precision.loc[normalized.str.fullmatch(r'[0-9]{4}年', na=False)] = 'year'
    precision.loc[normalized.str.fullmatch(r'[0-9]{4}年[0-9]{1,2}月', na=False)] = 'month'
    output_df = pd.DataFrame({
        'built_at': raw['built_at_text'].map(parse_year_month),
        'built_at_precision': precision,
    })
    processed_outputs['built_at_text'] = output_df
    return output_df.head(10)

process_built_at_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `built_at_text`.
check_processed_column('built_at_text')

#### `floor_text`

階, tầng của căn hộ. Có thể là một tầng cụ thể như 3階 hoặc range như 1-3階 đối với nhà nhiều tầng.

In [ ]:
# Kiểm tra raw column `floor_text` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'floor_text')

In [ ]:
# Function xử lý riêng cho `floor_text`: parse tầng đơn hoặc range tầng thành floor_min/floor_max/floor_number.
def process_floor_text():
    parsed = raw['floor_text'].map(parse_floor_range)
    floor_min = pd.Series(parsed.map(lambda value: value[0]), index=raw.index)
    floor_max = pd.Series(parsed.map(lambda value: value[1]), index=raw.index)
    output_df = pd.DataFrame({
        'floor_min': pd.array(floor_min, dtype='Int64'),
        'floor_max': pd.array(floor_max, dtype='Int64'),
        'floor_number': pd.array(floor_min.where(floor_min == floor_max), dtype='Int64'),
    })
    processed_outputs['floor_text'] = output_df
    return output_df.head(10)

process_floor_text()

In [ ]:
# Kiểm tra lại output sau xử lý của column `floor_text`.
check_processed_column('floor_text')

#### `building_floors`

階建, tổng số tầng của tòa nhà hoặc mô tả kiểu 3階/10階建. Dùng để biết căn hộ nằm trong tòa cao bao nhiêu tầng.

In [ ]:
# Kiểm tra raw column `building_floors` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'building_floors')

In [ ]:
# Function xử lý riêng cho `building_floors`: parse tổng số tầng tòa nhà từ 階建.
def process_building_floors():
    output_df = pd.DataFrame({'building_total_floors': pd.array(raw['building_floors'].map(parse_building_total_floors), dtype='Int64')})
    processed_outputs['building_floors'] = output_df
    return output_df.head(10)

process_building_floors()

In [ ]:
# Kiểm tra lại output sau xử lý của column `building_floors`.
check_processed_column('building_floors')

#### `total_units`

総戸数, tổng số căn/hộ trong tòa nhà nếu source cung cấp. Có thể phản ánh quy mô tòa nhà.

In [ ]:
# Kiểm tra raw column `total_units` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'total_units')

In [ ]:
# Function xử lý riêng cho `total_units`: parse 総戸数 từ dạng 123戸 sang số căn.
def process_total_units():
    output_df = pd.DataFrame({'total_units_count': pd.array(raw['total_units'].map(parse_total_units), dtype='Int64')})
    processed_outputs['total_units'] = output_df
    return output_df.head(10)

process_total_units()

In [ ]:
# Kiểm tra lại output sau xử lý của column `total_units`.
check_processed_column('total_units')

### Nhóm: Loại tòa nhà, kết cấu và hướng

#### `building_type`

建物種別, loại bất động sản như mansion/apartment. Ở Nhật マンション thường là chung cư/kết cấu kiên cố hơn apartment nhỏ.

In [ ]:
# Kiểm tra raw column `building_type` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'building_type')

In [ ]:
# Function xử lý riêng cho `building_type`: normalize loại tòa nhà và đổi sentinel thành null.
def process_building_type():
    output_df = pd.DataFrame({'building_type_clean': clean_categorical(raw['building_type'])})
    processed_outputs['building_type'] = output_df
    return output_df.head(10)

process_building_type()

In [ ]:
# Kiểm tra lại output sau xử lý của column `building_type`.
check_processed_column('building_type')

#### `structure`

構造, kết cấu tòa nhà như RC, SRC, thép, gỗ. Có thể ảnh hưởng chất lượng, cách âm, tuổi thọ và giá thuê.

In [ ]:
# Kiểm tra raw column `structure` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'structure')

In [ ]:
# Function xử lý riêng cho `structure`: normalize kết cấu tòa nhà và đổi sentinel thành null.
def process_structure():
    output_df = pd.DataFrame({'structure_clean': clean_categorical(raw['structure'])})
    processed_outputs['structure'] = output_df
    return output_df.head(10)

process_structure()

In [ ]:
# Kiểm tra lại output sau xử lý của column `structure`.
check_processed_column('structure')

#### `direction`

向き, hướng ban công/cửa chính của căn hộ như Nam, Đông, Tây. Có thể ảnh hưởng ánh sáng và giá.

In [ ]:
# Kiểm tra raw column `direction` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'direction')

In [ ]:
# Function xử lý riêng cho `direction`: normalize hướng căn hộ và đổi sentinel thành null.
def process_direction():
    output_df = pd.DataFrame({'direction_clean': clean_categorical(raw['direction'])})
    processed_outputs['direction'] = output_df
    return output_df.head(10)

process_direction()

In [ ]:
# Kiểm tra lại output sau xử lý của column `direction`.
check_processed_column('direction')

### Nhóm: Điều kiện thuê, bảo hiểm và bãi đỗ xe

#### `conditions`

条件, điều kiện thuê như pet, người ở, loại khách thuê, hoặc các điều kiện đặc biệt. Coverage không đầy đủ nên cần dùng thận trọng.

In [ ]:
# Kiểm tra raw column `conditions` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'conditions')

In [ ]:
# Function xử lý riêng cho `conditions`: normalize điều kiện thuê và đổi sentinel thành null.
def process_conditions():
    output_df = pd.DataFrame({'conditions_clean': clean_categorical(raw['conditions'])})
    processed_outputs['conditions'] = output_df
    return output_df.head(10)

process_conditions()

In [ ]:
# Kiểm tra lại output sau xử lý của column `conditions`.
check_processed_column('conditions')

#### `insurance`

損保, bảo hiểm tài sản/hỏa hoạn khi thuê nhà. Có thể ghi cần bảo hiểm, số tiền, và kỳ hạn như 1.8万円2年.

In [ ]:
# Kiểm tra raw column `insurance` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'insurance')

In [ ]:
# Function xử lý riêng cho `insurance`: parse yêu cầu bảo hiểm, số tiền và kỳ hạn bảo hiểm.
def process_insurance():
    parsed = raw['insurance'].map(parse_insurance)
    output_df = pd.DataFrame({
        'insurance_required': pd.array(parsed.map(lambda value: value[0]), dtype='boolean'),
        'insurance_jpy': pd.array(parsed.map(lambda value: value[1]), dtype='Int64'),
        'insurance_period_years': pd.array(parsed.map(lambda value: value[2]), dtype='Int64'),
    })
    processed_outputs['insurance'] = output_df
    return output_df.head(10)

process_insurance()

In [ ]:
# Kiểm tra lại output sau xử lý của column `insurance`.
check_processed_column('insurance')

#### `parking`

駐車場, thông tin bãi đỗ xe. Có thể ghi trong khuôn viên, gần đó, khoảng cách và phí theo tháng; dấu - chỉ là source không list parking rõ ràng.

In [ ]:
# Kiểm tra raw column `parking` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'parking')

In [ ]:
# Function xử lý riêng cho `parking`: parse thông tin bãi đỗ xe, loại, khoảng cách và phí tháng.
def process_parking():
    parsed = raw['parking'].map(parse_parking)
    output_df = pd.DataFrame({
        'parking_listed': pd.array(parsed.map(lambda value: value[0]), dtype='boolean'),
        'parking_type': parsed.map(lambda value: value[1]).astype('string'),
        'parking_distance_m': pd.array(parsed.map(lambda value: value[2]), dtype='Int64'),
        'parking_fee_jpy': pd.array(parsed.map(lambda value: value[3]), dtype='Int64'),
    })
    processed_outputs['parking'] = output_df
    return output_df.head(10)

process_parking()

In [ ]:
# Kiểm tra lại output sau xử lý của column `parking`.
check_processed_column('parking')

### Nhóm: Năng lượng và tiện ích dự kiến

#### `energy_efficiency`

エネルギー消費性能, hiệu suất tiêu thụ năng lượng. Snapshot hiện gần như chỉ có sentinel thiếu dữ liệu nên chưa có giá trị phân tích.

In [ ]:
# Kiểm tra raw column `energy_efficiency` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'energy_efficiency')

In [ ]:
# Function xử lý riêng cho `energy_efficiency`: normalize hiệu suất năng lượng; hiện chủ yếu là sentinel thiếu dữ liệu.
def process_energy_efficiency():
    output_df = pd.DataFrame({'energy_efficiency_clean': clean_categorical(raw['energy_efficiency'])})
    processed_outputs['energy_efficiency'] = output_df
    return output_df.head(10)

process_energy_efficiency()

In [ ]:
# Kiểm tra lại output sau xử lý của column `energy_efficiency`.
check_processed_column('energy_efficiency')

#### `insulation_performance`

断熱性能, hiệu suất cách nhiệt. Snapshot hiện gần như không có dữ liệu thực.

In [ ]:
# Kiểm tra raw column `insulation_performance` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'insulation_performance')

In [ ]:
# Function xử lý riêng cho `insulation_performance`: normalize hiệu suất cách nhiệt; hiện chủ yếu là sentinel thiếu dữ liệu.
def process_insulation_performance():
    output_df = pd.DataFrame({'insulation_performance_clean': clean_categorical(raw['insulation_performance'])})
    processed_outputs['insulation_performance'] = output_df
    return output_df.head(10)

process_insulation_performance()

In [ ]:
# Kiểm tra lại output sau xử lý của column `insulation_performance`.
check_processed_column('insulation_performance')

#### `estimated_utility_cost`

目安光熱費, chi phí điện/nước/gas tham khảo. Snapshot hiện gần như không có dữ liệu thực.

In [ ]:
# Kiểm tra raw column `estimated_utility_cost` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'estimated_utility_cost')

In [ ]:
# Function xử lý riêng cho `estimated_utility_cost`: normalize chi phí tiện ích dự kiến; hiện chủ yếu là sentinel thiếu dữ liệu.
def process_estimated_utility_cost():
    output_df = pd.DataFrame({'estimated_utility_cost_clean': clean_categorical(raw['estimated_utility_cost'])})
    processed_outputs['estimated_utility_cost'] = output_df
    return output_df.head(10)

process_estimated_utility_cost()

In [ ]:
# Kiểm tra lại output sau xử lý của column `estimated_utility_cost`.
check_processed_column('estimated_utility_cost')

### Nhóm: Free text cần giữ staging/audit

#### `remarks`

備考, ghi chú tự do của listing. Có thể chứa nhiều thông tin phụ nhưng noise cao và không nên đưa vào mart wide nếu chưa có NLP/rule parse.

In [ ]:
# Kiểm tra raw column `remarks` trước khi áp dụng rule.
inspect_column_values(analysis_raw, 'remarks')

In [ ]:
# Function xử lý riêng cho `remarks`: không parse nội dung tự do; tạo flag có ghi chú và độ dài text để audit.
def process_remarks():
    normalized = normalize_text_series(raw['remarks'])
    output_df = pd.DataFrame({
        'remarks_present': normalized.notna() & ~normalized.isin(NULL_SENTINELS),
        'remarks_text_length': normalized.str.len().astype('Int64'),
    })
    processed_outputs['remarks'] = output_df
    return output_df.head(10)

process_remarks()

In [ ]:
# Kiểm tra lại output sau xử lý của column `remarks`.
check_processed_column('remarks')

## Bước 6 - Sanity checks cho rule quan trọng

Cell này thay thế cho test Python rời. Nếu một assert fail, dừng lại và sửa rule trước khi xuất CSV.

In [ ]:
assert parse_jpy_value('8.15万円') == 81_500
assert parse_jpy_value('８，０００円') == 8_000
assert parse_jpy_value('-', sentinel_as_zero=True) == 0
assert parse_month_value('1.1ヶ月') == 1.1
assert parse_area_m2('25.66m2') == 25.66
assert parse_building_age('新築') == 0
assert parse_building_age('築99年以上') == 99
assert parse_floor_range('1-3階') == (1, 3)
assert parse_building_total_floors('平屋') == 1
assert parse_building_total_floors('3階/10階建') == 10
assert parse_year_month('2008年') == pd.Timestamp('2008-01-01')
assert parse_year_month('2026年7月') == pd.Timestamp('2026-07-01')
assert parse_move_in("'26年9月初旬", 2026)[2] == 'early'
assert _resolve_two_digit_year(76, 2026) == 1976
assert parse_insurance('1.8万円2年') == (True, 18_000, 2)
assert parse_parking('近隣200m22000円') == (True, 'nearby', 200, 22_000)
row = pd.Series({'task_id': 1, 'batch_id': 1, 'suumo_property_code': '1', 'station_access': '阪急宝塚線/十三駅 バス8分 (バス停)加島中 歩3分'})
station_records, station_failures = _station_rows_for_record(row)
assert station_failures == 0
assert station_records[0]['station_name'] == '十三'
assert station_records[0]['total_travel_minutes'] == 11
print('Sanity checks passed')

## Bước 7 - Clean data và tạo report

Bước này chạy sau khi đã xem raw column profile và quyết định xử lý. Không drop raw row; current listing là dataset riêng.

In [ ]:
cleaned, report = clean_suumo_records(raw)
current = build_current_listings(cleaned)
stations, station_failures = build_station_access(cleaned)
current_task_ids = set(current['task_id'].astype(int))
current_stations = stations[stations['task_id'].astype(int).isin(current_task_ids)].copy()

report.station_rows = len(stations)
report.station_parse_failures = station_failures

pd.DataFrame({
    'dataset': ['raw history', 'clean history', 'current listings', 'station access history', 'current station access'],
    'rows': [len(raw), len(cleaned), len(current), len(stations), len(current_stations)],
    'columns': [len(raw.columns), len(cleaned.columns), len(current.columns), len(stations.columns), len(current_stations.columns)],
})

## Bước 8 - Kiểm tra parse failures, hidden missing, và quality flags

Các bảng này cho biết rule nào còn không parse được, field nào có hidden missing nhiều, và record nào cần review thủ công thay vì tự sửa dữ liệu.

In [ ]:
display(pd.Series(report.parse_failure_counts, name='parse_failures').to_frame())
display(pd.Series(report.quality_flag_counts, name='quality_flags').to_frame())
sentinel_summary = (
    pd.Series(report.sentinel_counts, name='count')
      .to_frame()
      .assign(pct=lambda frame: (frame['count'] / len(raw) * 100).round(2))
)
display(sentinel_summary.head(40))

In [ ]:
safe_clean_columns = [
    'task_id', 'suumo_property_code', 'rent_jpy', 'management_fee_jpy',
    'base_monthly_cost_jpy', 'deposit_jpy', 'key_money_jpy',
    'exclusive_area_m2', 'rent_per_m2_jpy', 'layout_normalized',
    'building_age_years', 'floor_number', 'building_total_floors',
    'direction_clean', 'building_type_clean', 'structure_clean',
    'primary_station_name', 'primary_access_mode',
    'primary_total_travel_minutes', 'move_in_status', 'parking_listed',
    'quality_issue_count',
]
current[safe_clean_columns].head(20)

## Bước 9 - EDA sau cleaning

Phần này dùng current listings để xem phân phối số và category sau khi dữ liệu đã typed. Đây là nơi xác nhận rule có tạo ra giá trị hợp lý trước khi xuất CSV.

In [ ]:
numeric_columns = [
    'rent_jpy', 'management_fee_jpy', 'base_monthly_cost_jpy',
    'deposit_jpy', 'key_money_jpy', 'exclusive_area_m2',
    'rent_per_m2_jpy', 'building_age_years', 'primary_total_travel_minutes',
]
numeric = current[numeric_columns].apply(pd.to_numeric, errors='coerce')
numeric.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T

In [ ]:
category_columns = [
    'layout_normalized', 'direction_clean', 'building_type_clean',
    'structure_clean', 'transaction_type_clean', 'move_in_status',
    'parking_listed', 'contract_type',
]
for column in category_columns:
    print(f'\n--- {column} ---')
    display(current[column].value_counts(dropna=False).head(15).to_frame('count'))

In [ ]:
coverage_columns = [
    'rent_jpy', 'management_fee_jpy', 'deposit_jpy', 'key_money_jpy',
    'exclusive_area_m2', 'building_age_years', 'building_total_floors',
    'primary_station_name', 'primary_total_travel_minutes', 'total_units_count',
    'contract_type', 'insurance_jpy', 'parking_fee_jpy',
    'other_initial_costs_total_jpy', 'direction_clean', 'conditions_clean',
]
coverage = pd.DataFrame({
    'non_null': current[coverage_columns].notna().sum(),
    'coverage_pct': (current[coverage_columns].notna().mean() * 100).round(2),
}).sort_values('coverage_pct')
coverage

In [ ]:
review_columns = [
    'task_id', 'suumo_property_code', 'rent_jpy', 'exclusive_area_m2',
    'rent_per_m2_jpy', 'layout_normalized', 'building_age_years',
    'building_type_clean', 'primary_total_travel_minutes',
    'flag_rent_per_m2_outlier', 'flag_high_rent_small_area',
    'flag_future_completion',
]
review_candidates = current.loc[
    current['quality_issue_count'].astype('Int64') > 0,
    review_columns,
].sort_values('rent_per_m2_jpy', ascending=False)
review_candidates

## Bước 10 - Xuất CSV clean

Chỉ chạy sau khi các bảng kiểm tra ở trên hợp lý. Output gồm full history clean, current listings, station access dạng one-to-many, và report JSON để trace lại coverage/flags.

In [ ]:
for path in [CLEAN_PATH, CURRENT_PATH, STATION_PATH, CURRENT_STATION_PATH, REPORT_PATH]:
    path.parent.mkdir(parents=True, exist_ok=True)

cleaned.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')
current.to_csv(CURRENT_PATH, index=False, encoding='utf-8-sig')
stations.to_csv(STATION_PATH, index=False, encoding='utf-8-sig')
current_stations.to_csv(CURRENT_STATION_PATH, index=False, encoding='utf-8-sig')

payload = report.to_dict()
payload['outputs'] = {
    'raw_history': str(RAW_PATH.resolve()),
    'clean_history': str(CLEAN_PATH.resolve()),
    'current_listings': str(CURRENT_PATH.resolve()),
    'station_access': str(STATION_PATH.resolve()),
    'current_station_access': str(CURRENT_STATION_PATH.resolve()),
}
REPORT_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2, default=str) + '\n', encoding='utf-8')

print(f'Raw history: {len(raw):,} rows x {len(raw.columns)} columns -> {RAW_PATH.name}')
print(f'Clean history: {len(cleaned):,} rows x {len(cleaned.columns)} columns -> {CLEAN_PATH.name}')
print(f'Current listings: {len(current):,} rows -> {CURRENT_PATH.name}')
print(f'Station access history: {len(stations):,} rows -> {STATION_PATH.name}')
print(f'Current station access: {len(current_stations):,} rows -> {CURRENT_STATION_PATH.name}')
print(f'Report: {REPORT_PATH.name}')

## Sau notebook

Dùng `suumo_rentals_current.csv` cho phân tích cross-section hiện tại, `suumo_parser_records_clean.csv` cho lịch sử/thay đổi theo thời gian, và `suumo_station_access_current.csv` cho phân tích ga/tuyến. Khi rule ổn định, chuyển các derived columns sang dbt staging/intermediate models.